# Exercise 09 — The Capital Asset Pricing Model

MSc Finance · Investments · FHNW · Autumn 2026

Every exercise so far has taken expected returns as given. Exercise 05 fed them into an
optimizer and watched the weights explode; Exercise 06 estimated betas against an index
that was simply assumed to be the right one. Today we run the argument in the other
direction. Three companies, three investors, one risk-free asset — and the question is
not *which portfolio should I hold* but *what must expected returns be so that everyone
can hold what actually exists*.

There is no data file this week. The economy is fourteen numbers, and every result in
this notebook can be checked with a pen. That is the point: when the answer comes out
wrong you will know it is the reasoning, not the data.

**One change from the lecture.** Slide 16 set all pairwise correlations to zero so that
the arithmetic could be done at the whiteboard. Here $\rho = 0.20$ throughout. Nothing
in the theory changes. Almost every number does.

**How to work with this notebook.** The task text is here and on the exercise sheet.
Each code cell is a stub: the `# TODO` lines are the steps, in order. The setup cell
below is complete — run it and leave it alone. Tasks 1 to 3 are the exercise; Task 4 is
there for groups that finish early, and we walk through its result together whether or
not you ran it.

Run **Runtime → Restart and run all** before you trust any number in here. Nothing in
this notebook draws random numbers, so every group's output is identical to the last
decimal.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, save_results
setup_style()

### The economy

Three Swiss-listed companies, three investors, one risk-free asset in zero net supply.
All figures in CHF billion. This is slide 16, with one number changed.

The cell below is complete. Read it before you run it — the six lines after `RHO` are
the entire model.

In [ ]:
ASSETS = ["A", "B", "C"]

# --- the three companies ---------------------------------------------------
cap = pd.Series([250.0, 150.0, 100.0], index=ASSETS)   # market capitalisation
sd = pd.Series([0.20, 0.20, 0.20], index=ASSETS)       # volatility
w_m = cap / cap.sum()                                  # market portfolio weights

# --- the three investors ---------------------------------------------------
investors = pd.DataFrame(
    {"wealth": [200.0, 200.0, 100.0], "A": [5.0, 2.5, 1.25]},
    index=["I  pension fund", "II  private investor", "III  leveraged fund"])

rf = 0.02

# --- the one assumption we change against the lecture ----------------------
RHO = 0.20                                             # slide 16 used 0.0

corr = np.full((3, 3), RHO)
np.fill_diagonal(corr, 1.0)
Sigma = pd.DataFrame(np.outer(sd, sd) * corr, index=ASSETS, columns=ASSETS)

# The expected excess returns the lecture worked with. They belong to the rho = 0
# economy of slide 16. Whether they survive rho = 0.20 is Task 1.
mu_slide = pd.Series([0.05, 0.03, 0.02], index=ASSETS)

print("market weights\n" + (100 * w_m).round(1).to_string())
print("\ncovariance matrix\n" + Sigma.to_string())
print("\ntotal wealth", investors["wealth"].sum(), " total market cap", cap.sum())

## Task 1 — The lecture's numbers stop working

Slide 16 gave the three companies expected excess returns of 5 %, 3 % and 2 %, and
slide 18 showed that the three investors between them end up holding exactly
50 / 30 / 20 — the market portfolio. That was an equilibrium *for* $\rho = 0$.

Keep those three expected excess returns. Change nothing else except $\rho = 0.20$, and
ask what portfolio of risky assets an investor now wants.

Every mean-variance investor holds the tangency portfolio, and there is a closed form
for it — no optimizer needed:

$$w_{TP} \;\propto\; \Sigma^{-1}\bigl(E(R) - R_f\bigr), \qquad\text{then rescale so that }
\sum_i w_i = 1 .$$

Write it as a function, because you will want it again in Task 3. Do **not** invert
$\Sigma$ explicitly — `np.linalg.solve(Sigma, mu)` returns $\Sigma^{-1}\mu$ directly and
is both faster and better behaved.

Then put the tangency weights next to the market weights, and answer in one sentence:
if every investor in this economy wants the portfolio you have just computed, who holds
the rest of company C?

*Deliverable: the tangency weights against the market weights, as one printed table, and
that one sentence.*

In [ ]:
# TODO: tangency(mu, Sigma) -> weights proportional to inv(Sigma) @ mu, summing to 1
def tangency(mu, Sigma=Sigma):
    ...
# TODO: the tangency portfolio implied by the lecture's expected excess returns
comparison = pd.DataFrame({"market": 100 * w_m,
                           "wanted": 100 * w_tp_slide})
comparison["gap"] = comparison["wanted"] - comparison["market"]
print("percent of the risky portfolio\n" + comparison.round(1).to_string())

## Task 2 — Reverse the map

The CAPM's move is to refuse the question Task 1 asked. Instead of computing weights
from expected returns, it takes the weights as observed — they are the market
capitalisations, which are public — and asks what expected returns would make them
optimal.

Three steps, in this order.

**Step 1: the price of risk.** Each investor $j$ puts a share $y_j^* = \dfrac{E(R_M) -
R_f}{A_j \sigma_M^2}$ of their wealth in the risky portfolio. Total risky demand must
equal total market capitalisation, and total wealth equals total market capitalisation
too, because the risk-free asset is borrowing between these three investors and nets to
zero. Impose that, and the market's aggregate risk aversion $\bar{A}$ falls out:

$$\frac{1}{\bar{A}} \;=\; \sum_j \frac{W_j / W}{A_j} .$$

**Step 2: the market risk premium.** $E(R_M) - R_f = \bar{A}\,\sigma_M^2$, with
$\sigma_M^2 = w_M' \Sigma\, w_M$.

**Step 3: every individual asset.** $\beta_i = \dfrac{\operatorname{Cov}(R_i, R_M)}{\sigma_M^2}$,
where the whole vector of covariances is one matrix product $\Sigma w_M$, and then
$E(R_i) - R_f = \beta_i\bigl(E(R_M) - R_f\bigr)$.

Build the equilibrium table — covariance with the market, beta, expected excess return,
expected return — and then run the three checks that tell you it is an equilibrium:
the value-weighted average beta, each investor's $y^*$, and whether the tangency
portfolio of your new expected returns is the market portfolio.

Finally, plot the economy twice: in $(\sigma, E(R))$ space with the capital allocation
line through M, and in $(\beta, E(R))$ space with the security market line. Same three
assets, two coordinate systems.

*Deliverable: one equilibrium table, three checks that pass, and the two-panel figure.*

In [ ]:
# TODO: aggregate risk aversion from the three investors (step 1)
# TODO: market variance, market risk premium (step 2)
# TODO: covariances with the market, betas, equilibrium excess returns (step 3)
equilibrium = pd.DataFrame({"Cov(Ri,RM)": cov_m, "beta": beta,
                            "E(R)-Rf": 100 * mu_eq,
                            "E(R)": 100 * (rf + mu_eq)})
print(equilibrium.round(4).to_string())
print(f"\nA_bar = {A_bar:.4f}   sigma_M = {100 * sd_m:.4f}%   "
      f"E(RM)-Rf = {100 * prem_m:.4f}%   SR_M = {prem_m / sd_m:.4f}")

In [ ]:
# TODO: check 1 — the value-weighted average beta
print("value-weighted beta   ", round(w_m @ beta, 6))
# TODO: check 2 — each investor's optimal risky share
print("\ny* per investor\n" + (100 * y_star).round(1).to_string())
# TODO: check 3 — the tangency portfolio of the equilibrium expected returns
print("\ntangency vs market, percent\n" +
      pd.DataFrame({"tangency": 100 * tangency(mu_eq),
                    "market": 100 * w_m}).round(2).to_string())
# TODO: and the alphas, which is the same statement said differently
print("\nalphas, percent\n" + (100 * alpha_eq).round(6).to_string())

In [ ]:
fig1, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
ax = axes[0]
x = np.linspace(0, 0.26, 50)
# TODO: plot the capital allocation line -- from Rf, with the market's Sharpe ratio
ax.scatter(100 * sd_m, 100 * (rf + prem_m), s=90, color=FHNW["navy"],
           zorder=3, label="Market portfolio")
ax.annotate("M", (100 * sd_m, 100 * (rf + prem_m)),
            textcoords="offset points", xytext=(7, -12))
ax.scatter(100 * sd, 100 * (rf + mu_eq), s=70, color=FHNW["blue"],
           zorder=3, label="Individual assets")
for i in ASSETS:
    ax.annotate(i, (100 * sd[i], 100 * (rf + mu_eq[i])),
                textcoords="offset points", xytext=(7, -3))
ax.scatter(0, 100 * rf, s=50, color=FHNW["green"], zorder=3)
ax.annotate("$R_f$", (0, 100 * rf), textcoords="offset points", xytext=(7, -3))
ax.set_xlabel("Standard deviation, % p.a.")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)
ax = axes[1]
b = np.linspace(0, 1.4, 50)
# TODO: plot the security market line -- from Rf through the market at beta = 1
ax.scatter(1.0, 100 * (rf + prem_m), s=90, color=FHNW["navy"], zorder=3,
           label="Market portfolio")
ax.scatter(beta, 100 * (rf + mu_eq), s=70, color=FHNW["blue"], zorder=3,
           label="Individual assets")
for i in ASSETS:
    ax.annotate(i, (beta[i], 100 * (rf + mu_eq[i])),
                textcoords="offset points", xytext=(7, -3))
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)
fig1.tight_layout()

## Task 3 — Good news about C

Start from the equilibrium of Task 2. Analysts at one firm learn something good about
company C: its expected excess return is now **7.6 %**, four percentage points above the
equilibrium value you computed. Prices have not moved, so the market capitalisations are
still 250 / 150 / 100. Volatilities and correlations have not moved either, so no beta
changes.

Work out what that does to the economy.

1. The market portfolio contains C, so the market risk premium moves too. Recompute it,
   then the SML prediction and the alpha for each of the three assets. Check what the
   three alphas sum to when weighted by market capitalisation.
2. Recompute the tangency portfolio with `tangency()` from Task 1, and compare its
   Sharpe ratio with the market's.
3. Before the news, C traded at 100 with an expected payoff of 106.40. The news revises
   the expected payoff to 109.60. Prices adjust until $\alpha_C = 0$ again. At what
   price does that happen? Discount the new expected payoff at the return the security
   market line now requires of C.

Then redraw the two-panel figure of Task 2 with the new numbers, and answer: why do A
and B have non-zero alphas when no news arrived about either of them?

*Deliverable: the alpha table, the new tangency portfolio with both Sharpe ratios, the
restored price, and the figure.*

In [ ]:
# TODO: the revised expected excess returns, and the market premium they imply
# TODO: the SML prediction and the alpha for each asset -- betas are unchanged
news = pd.DataFrame({"E(R)-Rf": 100 * mu_news, "beta": beta,
                     "SML": 100 * sml, "alpha": 100 * alpha})
print(news.round(4).to_string())
print(f"\nmarket premium {100 * prem_m:.2f}% -> {100 * prem_news:.2f}%")
print(f"value-weighted alpha  {w_m @ alpha:.2e}")

In [ ]:
# TODO: the tangency portfolio of the revised expected returns
print("tangency vs market, percent\n" +
      pd.DataFrame({"tangency": 100 * w_tp,
                    "market": 100 * w_m}).round(2).to_string())
print(f"\nSR of the market   {sr_m_news:.4f}"
      f"\nSR of the tangency {sr_tp:.4f}"
      f"\ninformation ratio  {np.sqrt(sr_tp**2 - sr_m_news**2):.4f}")
# TODO: the price at which alpha_C returns to zero
print(f"\nC: required return {100 * required:.4f}%   "
      f"price 100.00 -> {price_new:.2f}  ({100 * (price_new / 100 - 1):+.2f}%)")

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
ax = axes[0]
x = np.linspace(0, 0.22, 50)
# TODO: two lines now -- through M, and the steeper one through the tangency
ax.scatter([100 * sd_m, 100 * sd_tp],
           [100 * (rf + prem_news), 100 * (rf + w_tp @ mu_news)],
           s=90, color=[FHNW["navy"], FHNW["blue"]], zorder=3)
ax.annotate("M", (100 * sd_m, 100 * (rf + prem_news)),
            textcoords="offset points", xytext=(8, -14))
ax.annotate("TP", (100 * sd_tp, 100 * (rf + w_tp @ mu_news)),
            textcoords="offset points", xytext=(-26, 4))
ax.set_xlabel("Standard deviation, % p.a.")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)
ax = axes[1]
b = np.linspace(0, 1.6, 50)
# TODO: the SML again -- same intercept, a different slope
for i in ASSETS:
    ax.plot([beta[i], beta[i]], [100 * (rf + sml[i]), 100 * (rf + mu_news[i])],
            color=FHNW["red"], lw=1.4, ls=":")
ax.scatter(beta, 100 * (rf + mu_news), s=70, color=FHNW["blue"], zorder=3,
           label="Individual assets")
ax.scatter(1.0, 100 * (rf + prem_news), s=90, color=FHNW["navy"], zorder=3,
           label="Market portfolio")
for i in ASSETS:
    ax.annotate(f"{i}: $\\alpha$ = {100 * alpha[i]:+.2f}%",
                (beta[i], 100 * (rf + mu_news[i])),
                textcoords="offset points", xytext=(7, -3), fontsize=9)
ax.set_xlabel(r"$\beta$")
ax.set_ylabel("Expected return, % p.a.")
ax.legend(loc="lower right", fontsize=9)
fig2.tight_layout()

## Task 4 (optional) — Turn the correlation dial

*For groups that finish Tasks 1 to 3 with time to spare. Nothing later depends on it,
and we work through the result together in the walkthrough either way.*

Task 2 built one equilibrium at $\rho = 0.20$, and the commentary compared it to the
lecture's $\rho = 0$. Do the whole sweep: for $\rho$ from 0 to 0.95, rebuild $\Sigma$,
recompute $\sigma_M$, the market risk premium, and the three betas.

Hold $\bar{A}$, the market weights and the individual volatilities fixed — none of them
is a function of correlation.

Plot the market risk premium and the three betas against $\rho$, and answer two
questions: what does the premium converge to as $\rho \to 1$, and what happens to the
cross-section of betas?

*Deliverable: one two-panel figure, and the two limiting values.*

In [ ]:
# TODO: rebuild the equilibrium for each rho on a grid
def equilibrium_at(rho):
    ...
rhos = np.linspace(0.0, 0.95, 96)
rows = []
for r in rhos:
    s_m, prem, b = equilibrium_at(r)
    rows.append({"sigma_M": s_m, "premium": prem, **b})
sweep = pd.DataFrame(rows, index=rhos)
print(sweep.loc[[0.0, 0.2, 0.6, 0.9], :].round(4).to_string())
fig3, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
axes[0].plot(rhos, 100 * sweep["premium"], color=FHNW["navy"], lw=2.0)
axes[0].axhline(100 * A_bar * 0.04, color=FHNW["green"], ls=":", lw=1.4)
axes[0].set_xlabel(r"Average correlation $\rho$")
axes[0].set_ylabel("Market risk premium, % p.a.")
for i, c in zip(ASSETS, ASSET_CYCLE):
    axes[1].plot(rhos, sweep[i], color=c, lw=2.0, label=i)
axes[1].axhline(1.0, color=FHNW["green"], ls=":", lw=1.4)
axes[1].set_xlabel(r"Average correlation $\rho$")
axes[1].set_ylabel(r"$\beta$")
axes[1].legend(fontsize=9)
fig3.tight_layout()

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.

In [ ]:
# TODO: collect the figures and tables into the two dicts save_results expects
if "fig3" in globals():                       # Task 4 was done
    figures["correlation_sweep"] = fig3
    tables["sweep"] = sweep
save_results(figures=figures, tables=tables, name="ex09")

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Everything in this notebook was an equilibrium we constructed. We chose $\bar{A}$, we
chose $\Sigma$, and the expected returns came out as whatever they had to be — so of
course the alphas were zero, and of course the market was the tangency portfolio. That
is a statement about the model's internal consistency and about nothing else.

The empirical question is the reverse one, and it is harder than it sounds: take a real
market, estimate real betas, and ask whether average realized returns line up along a
line through them. Lecture 10 first widens the model to several factors, Lecture 11 runs
the test properly, and Roll's critique — that the market portfolio we would need is not
observable — is waiting at the end of both.